In [ ]:
# 필요한 패키지 설치 및 확인
import sys
!{sys.executable} -m pip install --upgrade 'accelerate>=0.26.0'

# 설치 확인
import accelerate
print(f"✓ accelerate 버전: {accelerate.__version__}")


# 한국어 임베딩 모델 Fine-tuning

이 노트북은 KLUE RoBERTa 모델을 사용하여 한국어 문장 임베딩 모델을 학습합니다.
KLUE STS(Semantic Textual Similarity) 데이터셋을 활용하여 문장 간 의미적 유사도를 측정할 수 있는 모델을 만듭니다.


## 1. 임베딩 모델 초기화

**klue/roberta-base**를 기반으로 SentenceTransformer 모델을 만듭니다:
- `Transformer`: 사전학습된 RoBERTa 모델을 로드 (토큰 임베딩 생성)
- `Pooling`: 여러 토큰의 임베딩을 하나의 문장 임베딩으로 변환 (평균 풀링 사용)
- 이 두 레이어를 조합하여 최종 임베딩 모델을 생성합니다


In [3]:
from sentence_transformers import SentenceTransformer, models

# 1. Transformer 모델 로드
# - klue/roberta-base: 한국어에 최적화된 사전학습 모델 (HuggingFace)
# - 각 토큰을 벡터로 변환하는 역할 (예: "안녕" -> [0.1, 0.2, ..., 0.768])
word_embedding_model = models.Transformer("klue/roberta-base")

# 2. Pooling 레이어 생성
# - get_word_embedding_dimension(): RoBERTa의 출력 차원 (768차원)
# - pooling_mode_mean_tokens=True: 모든 토큰 임베딩의 평균을 계산
#   예: ["안녕", "하세요"] -> [[0.1,...], [0.2,...]] -> [0.15,...]
pooling_layer = models.Pooling(
  word_embedding_model.get_word_embedding_dimension(),
  pooling_mode_mean_tokens=True,
)

# 3. 최종 SentenceTransformer 모델 생성
# - Transformer + Pooling을 순차적으로 연결
# - 입력: 문장(텍스트) -> 출력: 고정 길이 벡터(768차원)
embedding_model = SentenceTransformer(
  modules=[word_embedding_model, pooling_layer]
)

Some weights of RobertaModel were not initialized from the model checkpoint at klue/roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 2. 데이터셋 로드

**KLUE STS** 데이터셋을 로드합니다:
- 한국어 문장 쌍과 그들 간의 유사도 점수(0~5)를 포함
- `train`: 학습용 데이터
- `validation`: 테스트용 데이터로 사용
- 예시: 두 문장이 얼마나 의미적으로 유사한지 점수로 표현


In [4]:
from datasets import load_dataset

# KLUE(Korean Language Understanding Evaluation) 벤치마크의 STS 데이터셋 로드
# - "klue": 데이터셋 이름
# - "sts": Semantic Textual Similarity 태스크
# - split="train": 학습용 데이터 (약 11,000개 문장 쌍)
# - split="validation": 검증용 데이터 (약 500개 문장 쌍)
klue_sts_train = load_dataset("klue", "sts", split="train")
klue_sts_test = load_dataset("klue", "sts", split="validation")

# 첫 번째 데이터 샘플 확인
# - sentence1, sentence2: 비교할 문장 쌍
# - labels.label: 0~5 사이의 유사도 점수 (5 = 완전히 같은 의미)
klue_sts_train[0]

{'guid': 'klue-sts-v1_train_00000',
 'source': 'airbnb-rtt',
 'sentence1': '숙소 위치는 찾기 쉽고 일반적인 한국의 반지하 숙소입니다.',
 'sentence2': '숙박시설의 위치는 쉽게 찾을 수 있고 한국의 대표적인 반지하 숙박시설입니다.',
 'labels': {'label': 3.7, 'real-label': 3.714285714285714, 'binary-label': 1}}

## 3. 학습/검증 데이터 분할

원래 학습 데이터를 다시 분할합니다:
- 90%는 학습용 (`klue_sts_train`)
- 10%는 검증용 (`klue_sts_eval`) - 학습 중 성능 모니터링용
- 테스트 데이터(`klue_sts_test`)는 별도로 유지


In [5]:
# 학습 데이터를 train/eval로 분할
# - test_size=0.1: 10%를 검증용으로 사용 (90% 학습, 10% 검증)
# - seed=42: 재현 가능하도록 랜덤 시드 고정
klue_sts_train = klue_sts_train.train_test_split(test_size=0.1, seed=42)

# 분할된 데이터를 각각의 변수에 할당
# - klue_sts_train: 약 9,900개 (실제 학습용)
# - klue_sts_eval: 약 1,100개 (학습 중 성능 모니터링용)
# - klue_sts_test: 약 500개 (최종 평가용, 위에서 로드한 validation)
klue_sts_train, klue_sts_eval = klue_sts_train["train"], klue_sts_train["test"]

## 4. 데이터 변환 함수

SentenceTransformer가 사용할 수 있는 형식으로 데이터를 변환합니다:
- `InputExample`: 두 문장과 유사도 레이블을 포함
- **중요**: 레이블을 5로 나누어 0~1 범위로 정규화 (모델 학습에 적합한 범위)


In [6]:
from sentence_transformers import InputExample

def prepare_sts_examples(dataset):
  """
  KLUE STS 데이터셋을 SentenceTransformer 학습 형식으로 변환
  
  Args:
    dataset: KLUE STS 데이터셋
  
  Returns:
    InputExample 리스트 (texts=[문장1, 문장2], label=정규화된 유사도)
  """
  examples = []

  for data in dataset:
    # InputExample 생성
    # - texts: 두 문장을 리스트로 전달
    # - label: 유사도 점수를 0~1 범위로 정규화 (원래 0~5 -> 5로 나눔)
    #   예: 3.7 / 5.0 = 0.74 (74% 유사함을 의미)
    examples.append(
      InputExample(
        texts=[data["sentence1"], data["sentence2"]],
        label=data["labels"]['label'] / 5.0
      )
    )
  return examples

## 5. 데이터 준비

각 데이터셋을 InputExample 형식으로 변환합니다:
- `train_examples`: 모델 학습용
- `eval_examples`: 학습 중 검증용
- `test_examples`: 최종 평가용


In [7]:
# 각 데이터셋을 InputExample 형식으로 변환
# - train_examples: 모델이 학습할 데이터 (약 9,900개)
# - eval_examples: 학습 중 성능을 확인할 데이터 (약 1,100개)  
# - test_examples: 최종 평가용 데이터 (약 500개, 학습에 전혀 사용 안 함)
train_examples = prepare_sts_examples(klue_sts_train)
eval_examples = prepare_sts_examples(klue_sts_eval)
test_examples = prepare_sts_examples(klue_sts_test)

## 6. DataLoader 생성

학습 데이터를 배치로 나누어 제공하는 DataLoader를 생성합니다:
- `batch_size=16`: 한 번에 16개의 문장 쌍을 처리
- `shuffle=True`: 매 에폭마다 데이터 순서를 섞어 학습 효과 향상


In [8]:
from torch.utils.data import DataLoader

# PyTorch DataLoader 생성: 학습 데이터를 배치 단위로 제공
# - batch_size=16: 한 번에 16개의 문장 쌍을 처리 (GPU 메모리에 따라 조정 가능)
# - shuffle=True: 매 에폭마다 데이터를 무작위로 섞음 (과적합 방지)
#   예: 1 epoch에 9,900/16 ≈ 619 step이 필요
train_dataloader = DataLoader(train_examples, batch_size=16, shuffle=True)

## 7. Evaluator 설정

모델 성능을 측정하는 평가자를 생성합니다:
- `EmbeddingSimilarityEvaluator`: 문장 임베딩 간 코사인 유사도를 계산하고 실제 레이블과 비교
- Pearson 상관계수와 Spearman 상관계수를 계산하여 성능 측정
- `eval_evaluator`: 학습 중 사용
- `test_evaluator`: 최종 평가용


In [9]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# 평가자(Evaluator) 생성: 모델 성능을 측정하는 도구
# - 작동 방식:
#   1. 두 문장의 임베딩을 생성
#   2. 코사인 유사도 계산 (예: 0.85)
#   3. 실제 레이블과 비교 (예: 0.74)
#   4. Pearson/Spearman 상관계수로 전체 성능 측정

# eval_evaluator: 학습 중 500 step마다 성능 체크용
eval_evaluator = EmbeddingSimilarityEvaluator.from_input_examples(eval_examples)

# test_evaluator: 최종 평가용 (학습 전/후 비교)
test_evaluator = EmbeddingSimilarityEvaluator.from_input_examples(test_examples)

## 8. 학습 전 성능 측정

Fine-tuning 하기 전 모델의 baseline 성능을 확인합니다:
- `pearson_cosine`: 0.35 (예측 유사도와 실제 유사도의 피어슨 상관계수)
- `spearman_cosine`: 0.36 (예측 유사도와 실제 유사도의 스피어만 상관계수)
- **값이 높을수록 좋음** (최대 1.0)
- 현재는 fine-tuning 전이라 성능이 낮음


In [10]:
# Fine-tuning 전 모델의 baseline 성능 측정
# - 사전학습된 RoBERTa는 일반적인 언어 이해는 가능하지만
#   문장 유사도 측정에는 최적화되지 않음
# - 예상 결과: pearson/spearman 약 0.35 (낮은 성능)
# - Fine-tuning 후 0.8 이상으로 향상 예상
test_evaluator(embedding_model)

{'pearson_cosine': 0.3477069870589581, 'spearman_cosine': 0.35560473197486514}

## 다음 단계: 모델 학습


In [11]:
from sentence_transformers import losses

num_epochs = 4
model_name = 'klue/roberta-base'
model_save_path = 'output/training_sts_' + model_name.replace("/", "_")
train_loss = losses.CosineSimilarityLoss(embedding_model)

embedding_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=eval_evaluator,
    epochs=num_epochs,
    warmup_steps=100,
    evaluation_steps=1000, 
    output_path=model_save_path,
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/Users/hyunjin/ai/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Pearson Cosine,Spearman Cosine
657,0.028100,No log,0.955904,0.908842
1000,0.007900,No log,0.960018,0.918010
1314,0.007900,No log,0.959828,0.917301
1971,0.005200,No log,0.961628,0.920503
2000,0.003400,No log,0.961348,0.920756
2628,0.002500,No log,0.961806,0.921866


In [12]:
from datasets import load_dataset

klue_mrc_train = load_dataset("klue", "mrc", split="train")
klue_mrc_train[0]


{'title': '제주도 장마 시작 … 중부는 이달 말부터',
 'context': '올여름 장마가 17일 제주도에서 시작됐다. 서울 등 중부지방은 예년보다 사나흘 정도 늦은 이달 말께 장마가 시작될 전망이다.17일 기상청에 따르면 제주도 남쪽 먼바다에 있는 장마전선의 영향으로 이날 제주도 산간 및 내륙지역에 호우주의보가 내려지면서 곳곳에 100㎜에 육박하는 많은 비가 내렸다. 제주의 장마는 평년보다 2~3일, 지난해보다는 하루 일찍 시작됐다. 장마는 고온다습한 북태평양 기단과 한랭 습윤한 오호츠크해 기단이 만나 형성되는 장마전선에서 내리는 비를 뜻한다.장마전선은 18일 제주도 먼 남쪽 해상으로 내려갔다가 20일께 다시 북상해 전남 남해안까지 영향을 줄 것으로 보인다. 이에 따라 20~21일 남부지방에도 예년보다 사흘 정도 장마가 일찍 찾아올 전망이다. 그러나 장마전선을 밀어올리는 북태평양 고기압 세력이 약해 서울 등 중부지방은 평년보다 사나흘가량 늦은 이달 말부터 장마가 시작될 것이라는 게 기상청의 설명이다. 장마전선은 이후 한 달가량 한반도 중남부를 오르내리며 곳곳에 비를 뿌릴 전망이다. 최근 30년간 평균치에 따르면 중부지방의 장마 시작일은 6월24~25일이었으며 장마기간은 32일, 강수일수는 17.2일이었다.기상청은 올해 장마기간의 평균 강수량이 350~400㎜로 평년과 비슷하거나 적을 것으로 내다봤다. 브라질 월드컵 한국과 러시아의 경기가 열리는 18일 오전 서울은 대체로 구름이 많이 끼지만 비는 오지 않을 것으로 예상돼 거리 응원에는 지장이 없을 전망이다.',
 'news_category': '종합',
 'source': 'hankyung',
 'guid': 'klue-mrc-v1_train_12759',
 'is_impossible': False,
 'question_type': 1,
 'question': '북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은?',
 'answers': {'answer_start': [478, 478]

In [13]:
from sentence_transformers import SentenceTransformer, models

sentence_model = SentenceTransformer("shangrilar/klue-roberta-base-klue-sts")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/744 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
from datasets import load_dataset

klue_mrc_train = load_dataset("klue", "mrc", split="train")
klue_mrc_test = load_dataset("klue", "mrc", split="validation")

df_train = klue_mrc_train.to_pandas()
df_test = klue_mrc_test.to_pandas()

df_train = df_train[['title', 'question', 'context']]
df_test = df_test[['title', 'question', 'context']]


In [16]:
def add_ir_context(df):
  irrelevant_contexts = []
  for idx, row in df.iterrows():
    title = row['title']
    # title이 같지 않은 행에서 무작위로 하나 선택
    irrelevant_contexts.append(df.query(f"title != '{title}'").sample(n=1)['context'].values[0])
  
  # 모든 행을 처리한 후 컬럼 추가
  df['irrelevant_context'] = irrelevant_contexts
  return df


df_train_ir = add_ir_context(df_train)
df_test_ir = add_ir_context(df_test)